In [ ]:
!pip install flask twilio


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


: 

In [ ]:
import requests
from twilio.rest import Client
import os
from dotenv import load_dotenv
from twilio.rest import Client

# --- Load .env file ---
load_dotenv()

# --- Twilio setup variables ---
ACCOUNT_SID = os.getenv("ACCOUNT_SID")
AUTH_TOKEN = os.getenv("AUTH_TOKEN")
TWILIO_NUMBER = os.getenv("TWILIO_NUMBER")
TO_NUMBER = os.getenv("TO_NUMBER")

print("Twilio number:", TWILIO_NUMBER)
print("To number:", TO_NUMBER)
print("Account SID loaded:", ACCOUNT_SID is not None)

# Make client
client = Client(ACCOUNT_SID, AUTH_TOKEN)

def get_weather():
    """Fetch weather with translation to Swahili (translate=true)."""
    url = "http://127.0.0.1:5001/weather/tanzania?translate=true"
    response = requests.get(url)
    return response.json()

def format_message(data):
    """Format the SMS messages (now in Swahili)."""
    parts = data["sms"]["next_3_days"]["message"].split(" | ")

    # --- Header ---
    header = parts[0]  # "Taarifa ya hali ya hewa..." (in Swahili)

    # --- Today ---
    today = parts[1] if len(parts) > 1 else ""

    # --- Next 3 days ---
    next3 = parts[2] if len(parts) > 2 else ""

    # --- Next 7 days ---
    next7 = parts[3] if len(parts) > 3 else ""

    # --- Final compact SMS (in Swahili) ---
    sms = f"{header} | {today} | {next3} | {next7}"

    return sms


# --- Send SMS ---
def send_sms(message):
    client.messages.create(
        body=message,
        from_=TWILIO_NUMBER,
        to=TO_NUMBER
    )


weather = get_weather()
print("Weather data fetched successfully")
sms_text = format_message(weather)
print("Sending SMS (in Swahili):", sms_text[:100], "...")
send_sms(sms_text)

Twilio number: +13509608232
To number: +32474060826
Account SID loaded: True
{'area_summary': {'next_3_days': {'dominant_weather': 'Thunderstorm', 'max_total_precipitation': 19.9, 'temp_max': 35.9, 'temp_min': 17.2}, 'next_7_days': {'dominant_weather': 'Thunderstorm', 'max_total_precipitation': 76.0, 'temp_max': 36.1, 'temp_min': 17.2}, 'today': {'dominant_weather': 'Thunderstorm', 'max_rain_risk_next_12h': 40, 'max_wind_peak_next_12h': 12.7, 'temp_max': 35.7, 'temp_min': 17.5}}, 'bounding_box': {'max_lat': -0.9853, 'max_lon': 40.4432, 'min_lat': -11.7613, 'min_lon': 29.3272}, 'input_points': [], 'meta': {'generated_at': '2026-03-17T11:45:29', 'sample_count': 5, 'scope': 'area', 'source': 'open-meteo', 'timezone': 'Africa/Dar_es_Salaam'}, 'sample_points': [{'daily_summary_next_3d': {'days_considered': 3, 'dominant_weather': 'Thunderstorm + hail', 'period_end': '2026-03-19', 'period_start': '2026-03-17', 'temp_max': 29.0, 'temp_min': 17.2, 'total_precipitation': 12.0}, 'daily_summary_ne